<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/ai-act-conformity/lessons/P01-L05-human-oversight/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/ai-act-conformity/lessons/P01-L05-human-oversight/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/ai-act-conformity/lessons/P01-L05-human-oversight/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/ai-act-conformity/lessons/P01-L05-human-oversight/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P01-L05 · Human oversight as a measurable property

**You will build:** six measurements over the Article 12 log you built in module 1 —
override rate by outcome, time to decision, four-eyes compliance where Article 14(5) bites,
and **automation bias**, the override rate as a function of the model's own confidence —
and then the control: an `oversight_gate()` that blocks an action whose verifications the
policy does not support, run over the log to find the decisions it would have stopped.

**Time:** ~75 minutes · **Runs on:** a laptop CPU, no download, no network
· **Prerequisites:** `T10-L01-ai-act-conformity-pack`, `P01-L01-article-12-logging`

Article 14 says a high-risk system must be designed so natural persons can *effectively*
oversee it. Almost every human-oversight plan you will ever read answers that with an org
chart and the sentence "all decisions are reviewed by a trained caseworker". A reviewer who
overrides two per cent of the time and never once disagrees with a high-confidence score
satisfies that sentence and oversees nothing.

By the end you will be able to:

1. Implement a decision view that folds module 1's chained log into one row per decision,
   in sequence order, keeping the evidence a naive fold destroys.
2. Implement override rate overall, by reviewer and by the model's recommendation, and
   explain which decisions have to be excluded from the denominator and why.
3. Implement a time-to-decision distribution that tells a clock anomaly apart from a
   reviewer who is too fast.
4. Implement an automation-bias curve and separate a reviewer whose override rate collapses
   to nothing from one whose rate merely declines, using each reviewer's own detectable floor.
5. Implement the Article 14(5) four-eyes check as a count of **distinct natural persons**, then
   `oversight_gate()`, the control that enforces it, and report every decision it would have
   blocked with each reason it failed on.

> **This is engineering, not legal advice.** The article numbers, the quoted wording and the
> dates are sourced in `claims.yaml` with their URLs and access dates. The thresholds below —
> a 30-decision support floor, a 10-second plausibility floor, a 5 % high-confidence override
> floor, the ban on self-verification — are **this lesson's modelling choices**, argued for
> where they appear. They are not statements about what any authority would accept. For a
> real system, read the Official Journal text and take professional advice.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import hashlib
import io
import json
import math
import sys
import traceback
from datetime import date, datetime, timedelta, timezone
from typing import Any, Callable

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__)

AS_OF = date(2026, 9, 22)
SYSTEM_ID = "loan-copilot"
GENESIS_HASH = "0" * 64

# Every threshold this lesson applies, in one place, because a threshold buried in a function
# is a threshold nobody argued with. Each is defended where it is first used.
MIN_SUPPORT = 30              # decisions below which a per-reviewer rate is not reported
MIN_PLAUSIBLE_SECONDS = 10    # faster than this and nobody read the case
HIGH_CONFIDENCE_FLOOR = 0.05  # a high-confidence override rate at or under this is a stamp
CONFIDENCE_SPLIT = 0.80       # the cut between the "low" and "high" confidence halves
CONFIDENCE_BINS = (0.50, 0.65, 0.80, 0.90, 1.01)   # four bins, half-open, last edge inclusive

# The two normal quantiles the detectable-difference floor uses. Same pair as P01-L04, and
# asserted rather than trusted: the cell below recomputes what they integrate to.
Z_ALPHA_95 = 1.959963985      # two-sided 95%: P(Z <= z) = 0.975
Z_BETA_80 = 0.841621234       # 80% power:     P(Z <= z) = 0.800


def normal_cdf(z: float) -> float:
    """The standard normal CDF, from math.erf. Given to you; not graded."""
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))


print(f"as of {AS_OF.isoformat()} · system {SYSTEM_ID}")
print(f"Phi(Z_ALPHA_95) = {normal_cdf(Z_ALPHA_95):.6f}  (must be 0.975000)")
print(f"Phi(Z_BETA_80)  = {normal_cdf(Z_BETA_80):.6f}  (must be 0.800000)")
assert abs(normal_cdf(Z_ALPHA_95) - 0.975) < 1e-6
assert abs(normal_cdf(Z_BETA_80) - 0.800) < 1e-6

_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("decision_view",),
    "exercise 2": ("override_rates",),
    "exercise 3": ("time_to_decision",),
    "exercise 4": ("four_eyes_report",),
    "exercise 5": ("automation_bias",),
    "exercise 6": ("oversight_gate", "gate_report"),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 5"] -> "exercise 5 (automation_bias)"; several -> "exercises 1, 2 and 5"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the other five. A demo names the exercises it
    `needs`: until each has passed its check, the demo says which one it is waiting for and
    skips. Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board
    at the foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script
    run non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def parse_ts(stamp: str) -> datetime:
    """Parse one of this log's ISO-8601 UTC timestamps. Given to you; not graded.

    Example:
        >>> parse_ts("2026-09-22T12:00:00+00:00").hour
        12
    """
    return datetime.fromisoformat(stamp.replace("Z", "+00:00"))

## 1. What Article 14 asks for, and what a log can see of it

Article 14(1) is a *design* duty: the system must be built so that natural persons can
effectively oversee it. Article 14(4) lists what those persons must be enabled to do, and
14(4)(b) names the failure mode by name — "the possible tendency of automatically relying or
over-relying on the output produced by a high-risk AI system (automation bias)". Article
14(5) adds a hard rule for one family of systems: no action on an identification unless it
has been separately verified and confirmed by at least two natural persons.

Three of those duties are properties of an interface — a stop button, an interpretation tool,
a limitations page — and a log cannot see them. Five of them leave a trace. The table below
says which is which, and the count under it is computed, not typed.

In [ ]:
DUTIES = {
    "art14_1_effective_oversight":
        "Article 14(1) — designed so natural persons can effectively oversee it while in use",
    "art14_4_a_monitor":
        "Article 14(4)(a) — understand capacities and limitations, and monitor for anomalies",
    "art14_4_b_automation_bias":
        "Article 14(4)(b) — remain aware of the tendency to over-rely on the output",
    "art14_4_c_interpret":
        "Article 14(4)(c) — correctly interpret the output, given the tools available",
    "art14_4_d_override":
        "Article 14(4)(d) — decide not to use, or to disregard, override or reverse the output",
    "art14_4_e_stop":
        "Article 14(4)(e) — intervene, or halt the system through a 'stop' button",
    "art14_5_two_persons":
        "Article 14(5) — Annex III point 1(a): no action on an identification unless separately "
        "verified and confirmed by at least two natural persons",
    "art26_2_competence":
        "Article 26(2) — the deployer assigns oversight to natural persons with the necessary "
        "competence, training and authority",
}

# Which duty each exercise in this lesson turns into a number. None means "a log cannot see
# this one" — and saying so is part of the artefact, because an oversight plan that claims to
# have measured the stop button has not been read by anybody.
MEASURED_BY = {
    "art14_1_effective_oversight": "override_rates + time_to_decision",
    "art14_4_a_monitor": None,
    "art14_4_b_automation_bias": "automation_bias",
    "art14_4_c_interpret": None,
    "art14_4_d_override": "override_rates",
    "art14_4_e_stop": None,
    "art14_5_two_persons": "four_eyes_report + oversight_gate",
    "art26_2_competence": "time_to_decision (a proxy only — see section 5)",
}

_measured = [k for k, v in MEASURED_BY.items() if v]
print(f"{len(DUTIES)} duties listed · {len(_measured)} this lesson turns into a number · "
      f"{len(DUTIES) - len(_measured)} are interface properties a log cannot see")
for _duty, _text in DUTIES.items():
    _mark = MEASURED_BY[_duty] or "not measurable from a log"
    print(f"  [{'x' if MEASURED_BY[_duty] else ' '}] {_duty:30s} {_mark}")
print("\n" + DUTIES["art14_5_two_persons"])

**Read 14(5) twice before you generalise it.** It binds the Annex III **point 1(a)** systems
— remote biometric identification — and it carves out law enforcement, migration, border
control and asylum where Union or national law considers it disproportionate. It is *not* a
two-person rule for every high-risk system, and an oversight plan that applies it everywhere
is wrong in an expensive direction.

The fictional system here, `loan-copilot`, scores credit applications. Most of its decisions
are ordinary credit files. A minority arrive through a remote onboarding flow that identifies
the applicant biometrically, and *those* decisions sit under Annex III point 1(a). So the
two-person rule applies to part of the log and not the rest, which is exactly the shape a
real deployer has to implement — and exactly the shape a `for` loop over the whole log gets
wrong.

In [ ]:
# The deployer's own oversight policy. The two-person figure for remote_biometric comes from
# Article 14(5); everything else here is the deployer's choice, and labelled as such.
POLICY = {
    "min_distinct_verifiers": {"remote_biometric": 2,   # Article 14(5)
                               "credit_file": 1},       # deployer policy, not the Act
    "min_seconds": MIN_PLAUSIBLE_SECONDS,               # deployer policy
    "require_outcome": True,                            # deployer policy
    "forbid_self_verification": True,                   # deployer policy: see section 8
}
print(json.dumps(POLICY, indent=2, sort_keys=True))

## 2. The log

This is module 1's log, unchanged: an append-only list of entries, each carrying `seq`, the
`event`, `prev_hash` and an `entry_hash` over all three. You wrote `canonical_bytes` and
`append_event` there; they are given back to you here so this lesson starts from a log
rather than from a hashing exercise.

The events are module 1's too — `inference_requested`, `match_found`, `human_verification`,
`decision_recorded`, plus the system-level `model_deployed` — with three payload fields this
module needs and module 1 did not use: `recommendation` (what the model proposed),
`identification_basis` (which decides whether 14(5) applies) and `submitted_by`.

Nothing is loaded from disk and nothing is downloaded. The whole stream is built below from
one seed, so two runs of this notebook agree to the last digit.

In [ ]:
def canonical_bytes(obj: Any) -> bytes:
    """Module 1's canonical serialisation. Given to you here; you built it in P01-L01."""
    return json.dumps(obj, sort_keys=True, separators=(",", ":")).encode("utf-8")


def append_event(log: list, event: dict) -> dict:
    """Module 1's chained append. Given to you here; you built it in P01-L01."""
    seq = len(log)
    prev = log[-1]["entry_hash"] if log else GENESIS_HASH
    digest = hashlib.sha256(
        canonical_bytes({"seq": seq, "prev_hash": prev, "event": event})).hexdigest()
    entry = {"seq": seq, "event": event, "prev_hash": prev, "entry_hash": digest}
    log.append(entry)
    return entry


RNG = np.random.default_rng(20260922)
_ANCHOR = datetime(2026, 9, 22, 12, 0, 0, tzinfo=timezone.utc)

REVIEWERS = ("u:marlies", "u:tomasz", "u:ines", "u:pieter")
SUPERVISOR = "u:sanne"                # co-signs the two-person decisions and nothing else
_COUNTS = {"u:marlies": 480, "u:tomasz": 430, "u:ines": 430, "u:pieter": 21}
# Each reviewer's probability of overriding, per confidence bin. You are not told these; the
# whole point of the lesson is to recover their shape from the log.
_P_OVERRIDE = {"u:marlies": (0.46, 0.31, 0.030, 0.005),
               "u:tomasz": (0.20, 0.19, 0.21, 0.20),
               "u:ines": (0.50, 0.40, 0.22, 0.13),
               "u:pieter": (0.30, 0.25, 0.20, 0.15)}
_REC_FACTOR = {"approve": 0.65, "decline": 1.65, "refer": 1.0}
_MEDIAN_SECONDS = {"u:marlies": 38.0, "u:tomasz": 190.0, "u:ines": 125.0, "u:pieter": 300.0}

_PLAN: list = []
for _rid, _n in _COUNTS.items():
    _PLAN += [_rid] * _n
_PLAN = [_PLAN[int(k)] for k in RNG.permutation(len(_PLAN))]
N_DECISIONS = len(_PLAN)


def _ts(offset_seconds: float) -> str:
    """An ISO-8601 UTC timestamp `offset_seconds` after the fixture's anchor."""
    return (_ANCHOR + timedelta(seconds=float(offset_seconds))).isoformat().replace(
        "+00:00", "Z")


def confidence_bin(confidence: float, bins: tuple = CONFIDENCE_BINS) -> int:
    """Which half-open bin `confidence` falls in. Given to you; not graded.

    Example:
        >>> confidence_bin(0.50), confidence_bin(0.80), confidence_bin(1.0)
        (0, 2, 3)
    """
    for k in range(len(bins) - 1):
        if bins[k] <= confidence < bins[k + 1]:
            return k
    return len(bins) - 2


def _build_stream() -> list:
    """Build the event stream. Every planted defect is listed here, in the open."""
    events: list = []

    def emit(offset, event_type, actor, payload, decision_id=None):
        event = {"ts": _ts(offset), "event_type": event_type, "actor": actor,
                 "system_id": SYSTEM_ID, "payload": payload}
        if decision_id is not None:
            event["decision_id"] = decision_id
        events.append(event)

    emit(-86400 * 120, "model_deployed", "mlops:release",
         {"model_version": "loan-risk-2026.04", "training_run": "TR-913",
          "approved_by": "u:ehsan"})

    scope = [i for i in range(N_DECISIONS) if i % 5 == 0]   # the Annex III 1(a) decisions
    single_verifier = set(scope[3::14])    # in scope, one verifier: a plain 14(5) breach
    dup_verifier = set(scope[7::23])       # in scope, the SAME person recorded twice
    no_verification = set(scope[5::37])    # in scope, no human_verification event at all
    skewed_pair = set(scope[9::53])        # two verifications, the clock stepped backwards
    unverified_out = set(range(3, N_DECISIONS, 101)) - set(scope)   # out of scope, unverified
    too_fast = set(range(7, N_DECISIONS, 113))     # under the plausibility floor
    clock_back = {211, 733, 1099}                  # use_end recorded BEFORE use_start
    unclosed = {58, 302, 519, 744, 961, 1188, 1301}   # no decision_recorded event
    unscored = {131, 407, 688, 1024, 1255}            # no match_found: no model recommendation
    # Some of these overlap the four-eyes defects on purpose: a decision can fail the policy
    # in more than one way, and a gate report that counts one reason per decision hides it.
    self_verified = (set(range(23, N_DECISIONS, 167))      # submitter verified their own case
                     | set(sorted(single_verifier)[:4]) | set(sorted(dup_verifier)[:3]))

    clock = 0.0
    for i, rid in enumerate(_PLAN):
        did = f"D-{2000 + i}"
        in_scope = (i % 5 == 0)
        basis = "remote_biometric" if in_scope else "credit_file"
        confidence = float(np.round(RNG.uniform(0.50, 1.00), 4))
        bin_index = confidence_bin(confidence)
        rec = ("approve", "decline", "refer")[int(RNG.choice(3, p=[0.50, 0.35, 0.15]))]
        overridden = bool(RNG.random() < _P_OVERRIDE[rid][bin_index] * _REC_FACTOR[rec])
        if overridden:
            alts = [o for o in ("approve", "decline", "refer") if o != rec]
            outcome = alts[int(RNG.integers(2))]
        else:
            outcome = rec

        submitted_by = rid if i in self_verified else "u:intake"
        clock += float(RNG.uniform(120.0, 900.0))
        start = clock
        emit(start, "inference_requested", "svc:scoring",
             {"use_start": _ts(start), "model_version": "loan-risk-2026.04",
              "input_reference": f"APP-{60000 + i}", "submitted_by": submitted_by,
              "identification_basis": basis,
              "reference_database": "credit-bureau-nl-2026Q3"}, decision_id=did)
        if i not in unscored:
            emit(start + 2, "match_found", "svc:scoring",
                 {"input_reference": f"APP-{60000 + i}", "match_score": confidence,
                  "recommendation": rec,
                  "reference_database": "credit-bureau-nl-2026Q3"}, decision_id=did)

        if i in too_fast:
            seconds = float(np.round(RNG.uniform(1.5, 8.5), 1))
        else:
            seconds = float(np.round(
                np.exp(np.log(_MEDIAN_SECONDS[rid]) + RNG.normal(0.0, 0.55)), 1))

        if i in no_verification or (i in unverified_out and not in_scope):
            verifiers: list = []
        elif i in dup_verifier:
            verifiers = [rid, rid]
        elif i in single_verifier:
            verifiers = [rid]
        elif in_scope:
            verifiers = [rid, SUPERVISOR]
        else:
            verifiers = [rid]

        if verifiers and i in skewed_pair and len(verifiers) == 2:
            # Two SEPARATE verification events, and the clock stepped back between them: the
            # second signature carries the earlier timestamp. Order by ts and they swap.
            emit(start + seconds - 5, "human_verification", "ui:caseworker",
                 {"verifier_ids": [verifiers[0]]}, decision_id=did)
            emit(start + seconds - 40, "human_verification", "ui:caseworker",
                 {"verifier_ids": [verifiers[1]]}, decision_id=did)
        elif verifiers:
            emit(start + seconds, "human_verification", "ui:caseworker",
                 {"verifier_ids": list(verifiers)}, decision_id=did)

        if i not in unclosed:
            end = start - abs(seconds) if i in clock_back else start + seconds
            emit(end, "decision_recorded", "svc:scoring",
                 {"outcome": outcome, "use_end": _ts(end)}, decision_id=did)

    return events


EVENT_STREAM = _build_stream()
LOG: list = []
for _event in EVENT_STREAM:
    append_event(LOG, _event)

_SYSTEM_LEVEL = sum(1 for e in EVENT_STREAM if "decision_id" not in e)
print(f"{len(LOG)} log entries · {N_DECISIONS} decisions · {_SYSTEM_LEVEL} system-level events")
print(f"{sum(1 for i in range(N_DECISIONS) if i % 5 == 0)} of those decisions arrived through "
      "the remote biometric onboarding flow, so Article 14(5) reaches them")
print(json.dumps(LOG[3], indent=2)[:520])

## 3. Exercise 1 — the decision view

Every measurement below is a `for` loop over decisions, not over events, so the first thing
to build is the fold: one record per `decision_id`, assembled from every entry that carries
it, **in sequence order**.

Two traps, both inherited from module 1 and both planted in the stream above.

* **Order by `seq`, never by `ts`.** Three decisions here have two separate verification
  events where the clock stepped backwards between them. Sort by timestamp and the two
  signatures come back in the wrong order.
* **Do not deduplicate `verifier_ids` here.** One person recorded twice is the single most
  common way a four-eyes rule is defeated in production, and a fold that quietly collapses
  `["u:marlies", "u:marlies"]` to one entry destroys the only evidence of it. Keep the raw
  list; the checks that follow are the ones that count *distinct* persons.

<details><summary>💡 Hint 1 — what to think about</summary>

What would sorting by timestamp do to the two verification events whose clock stepped
backwards between them — and what evidence vanishes if a person recorded twice becomes one
entry? Then two quieter traps: some entries belong to no decision at all, and a later event
that carries nothing (None, an empty string, an empty list) must not wipe out a real value.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Walk the entries in ascending `seq` and skip any whose event has no `decision_id`. The first
time you meet an id, start a record with every key present (an empty list for the
verifiers) and note its `seq`; every time, move `last_seq` on. Copy a scalar field only when
the payload holds something real, turn the match score into a float, and append each
non-empty verifier id as you meet it, never de-duplicating. When the walk is done, work out
each duration from the two timestamps where both exist: end minus start, sign kept.
</details>

In [ ]:
def decision_view(log: list) -> dict:
    """Fold the chained log into one record per decision, in sequence order.

    Walk `log` in ascending `seq`. Skip entries whose event has no "decision_id" — those are
    system-level events such as `model_deployed`, and they must not crash you or appear as a
    decision. For each decision build a record with exactly these keys:

      "decision_id"       the id
      "first_seq"         the seq of the first entry that mentions it
      "last_seq"          the seq of the last entry that mentions it
      "model_version"     from the payload, or None
      "confidence"        float(payload["match_score"]) from `match_found`, or None
      "recommendation"    payload["recommendation"], or None
      "identification_basis"  payload["identification_basis"], or None
      "submitted_by"      payload["submitted_by"], or None
      "verifier_ids"      every id from every "verifier_ids" payload, in seq order,
                          WITH duplicates kept and empty strings dropped
      "outcome"           payload["outcome"], or None
      "use_start"         payload["use_start"], or None
      "use_end"           payload["use_end"], or None
      "duration_seconds"  (parse_ts(use_end) - parse_ts(use_start)).total_seconds() when both
                          are present, else None. It MAY come out negative: three decisions
                          here recorded an end before their start, and hiding that with abs()
                          turns a broken clock into a fast reviewer.

    A later entry overwrites an earlier one for the scalar fields, but a value that is None,
    "" or [] never overwrites anything — those are what a half-built pipeline writes when it
    has nothing to say.

    Returns: dict mapping decision_id -> record dict, in first-appearance (seq) order.

    Example:
        >>> view = decision_view(LOG)
        >>> rec = view["D-2000"]
        >>> rec["identification_basis"], rec["recommendation"] is not None
        ('remote_biometric', True)
        >>> sorted(rec) == sorted(["decision_id", "first_seq", "last_seq", "model_version",
        ...                        "confidence", "recommendation", "identification_basis",
        ...                        "submitted_by", "verifier_ids", "outcome", "use_start",
        ...                        "use_end", "duration_seconds"])
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


_SCALAR_FIELDS = ("model_version", "submitted_by", "identification_basis",
                  "outcome", "use_start", "use_end")


def _check_decision_view() -> None:
    view = decision_view(LOG)
    assert isinstance(view, dict), f"decision_view returned {type(view).__name__}, want a dict"
    assert len(view) == N_DECISIONS, (
        f"got {len(view)} decisions, expected {N_DECISIONS} — if you have one more, you let "
        "the system-level model_deployed event in; it has no decision_id at all"
    )
    first = view["D-2000"]
    assert first["first_seq"] == 1, (
        f"D-2000 starts at seq {first['first_seq']}, expected 1 — seq 0 is model_deployed"
    )
    assert first["last_seq"] > first["first_seq"]
    assert first["identification_basis"] == "remote_biometric", (
        "identification_basis is in the inference_requested payload; fold it like the other "
        "scalar fields"
    )
    dup = [r for r in view.values()
           if len(r["verifier_ids"]) != len(set(r["verifier_ids"]))]
    assert dup, (
        "no decision came back with a repeated verifier — you deduplicated inside "
        "decision_view. Keep the raw list: section 6 needs to see the repeat to call it one"
    )
    unscored = [r for r in view.values() if r["confidence"] is None]
    assert len(unscored) == 5 and all(r["recommendation"] is None for r in unscored), (
        f"{len(unscored)} decisions have no confidence, expected 5 — those are the ones with "
        "no match_found event, and their recommendation must be None too"
    )
    negative = [r for r in view.values()
                if r["duration_seconds"] is not None and r["duration_seconds"] < 0]
    assert len(negative) == 3, (
        f"{len(negative)} decisions have a negative duration, expected 3 — do not take abs(); "
        "a clock that stepped backwards is not a fast reviewer"
    )
    missing = [r for r in view.values() if r["duration_seconds"] is None]
    assert len(missing) == 7 and all(r["outcome"] is None for r in missing), (
        f"{len(missing)} decisions have no duration, expected 7 — those never closed, so they "
        "have no use_end and no outcome"
    )
    print(f"decision view: {len(view)} decisions, {len(dup)} with a repeated verifier, "
          f"{len(negative)} with a backwards clock, {len(missing)} never closed")


_try("exercise 1", _check_decision_view)

## 4. Exercise 2 — override rate, and what has to leave the denominator

An **override** is a decision whose recorded outcome differs from what the model recommended.
The rate looks like one number and is really three, because three kinds of decision cannot
contribute to it and each has to be counted somewhere else instead:

* a decision that **never closed** has no outcome, so nobody overrode anything yet;
* a decision with **no model recommendation** — no `match_found` event — cannot be compared
  against the model at all, and calling it an override because `outcome != None` inflates
  every rate in the report;
* a decision with **no verifier** was not reviewed, so it belongs to no reviewer. It is the
  most important number on the page and the easiest one to lose.

Apply those three in that order, and report the counts you removed alongside the rate.

<details><summary>💡 Hint 1 — what to think about</summary>

Each excluded decision lands in exactly one counter, and the order you test the three
reasons decides which — a decision that never closed AND was never scored belongs to the
first. Then, for the breakdowns: whose call is the by-recommendation table measuring
disagreement with, the model's or the reviewer's? And how many times should one decision
count for a person whose id appears on it twice?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Test the exclusions in the docstring's order and stop at the first that applies. For each
survivor decide override-or-not once, then add it to the overall tally, to the tally of
every DISTINCT verifier on it, and to the tally keyed on what the model recommended. Turn
tallies into rates at the end (None for an empty one), and mark each reviewer supported
against the `min_support` argument, inclusive. Nobody below the floor is dropped.
</details>

In [ ]:
def override_rates(view: dict, min_support: int = MIN_SUPPORT) -> dict:
    """Override rate overall, per reviewer, and per model recommendation.

    Walk the records. In this order:
      1. outcome is None                -> count in "undecided", skip
      2. recommendation is None         -> count in "unscored", skip
      3. verifier_ids is empty          -> count in "unverified_decided", skip
    Everything that survives is a decided, scored, reviewed decision, and it is an override
    when outcome != recommendation.

    "overall"   {"n", "n_overrides", "rate"} over the survivors; rate is None when n == 0.
    "by_reviewer"  {reviewer_id: {"n", "n_overrides", "rate", "supported"}} — a decision counts
                once for each DISTINCT verifier on it, so the same person recorded twice on one
                decision is one decision for that person, not two. "supported" is n >=
                min_support; never drop an unsupported reviewer from the report.
    "by_outcome"   {recommendation: {"n", "n_overrides", "rate"}} keyed on what the MODEL
                recommended, not on what the human recorded. Keyed on the outcome instead, the
                override rate of "approve" becomes the rate at which people override INTO
                approve, which is a different question with the same shape.

    Returns: dict with keys "overall", "by_reviewer", "by_outcome", "unverified_decided",
    "undecided", "unscored".

    Example:
        >>> rep = override_rates(decision_view(LOG))
        >>> rep["undecided"], rep["unscored"], rep["unverified_decided"]
        (7, 5, 18)
        >>> round(rep["overall"]["rate"], 3)
        0.252
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_override_rates() -> None:
    rep = override_rates(decision_view(LOG))
    assert set(rep) == {"overall", "by_reviewer", "by_outcome", "unverified_decided",
                        "undecided", "unscored"}, f"keys came back {sorted(rep)}"
    assert (rep["undecided"], rep["unscored"]) == (7, 5), (
        f"undecided/unscored came back {(rep['undecided'], rep['unscored'])}, expected (7, 5) "
        "— apply the exclusions in order: no outcome first, then no recommendation"
    )
    assert rep["unverified_decided"] == 18, (
        f"unverified_decided is {rep['unverified_decided']}, expected 18 — these are decided, "
        "scored decisions with an EMPTY verifier list, and they belong to no reviewer"
    )
    assert rep["overall"]["n"] == 1331, (
        f"overall n is {rep['overall']['n']}, expected 1331 = {N_DECISIONS} - 7 - 5 - 18"
    )
    assert set(rep["by_reviewer"]) == set(REVIEWERS) | {SUPERVISOR}, (
        f"by_reviewer holds {sorted(rep['by_reviewer'])} — every verifier in the log belongs "
        "there, including the one below min_support"
    )
    assert rep["by_reviewer"]["u:pieter"]["supported"] is False, (
        "u:pieter reviewed fewer than MIN_SUPPORT decisions, so his rate is reported but "
        "flagged unsupported; dropping him would hide a reviewer nobody can evaluate"
    )
    assert set(rep["by_outcome"]) == {"approve", "decline", "refer"}
    spread = (rep["by_outcome"]["decline"]["rate"] - rep["by_outcome"]["approve"]["rate"])
    assert spread > 0.20, (
        f"decline minus approve override rate is {spread:.3f}; in this log it is well over "
        "0.20. A flat by_outcome table usually means you keyed on the recorded outcome "
        "instead of on the model's recommendation"
    )
    print(f"overall override rate {rep['overall']['rate']:.1%} over {rep['overall']['n']} "
          f"decisions · {rep['undecided']} undecided, {rep['unscored']} unscored, "
          f"{rep['unverified_decided']} never reviewed by anyone")
    for _rid in sorted(rep["by_reviewer"]):
        _cell = rep["by_reviewer"][_rid]
        _flag = "" if _cell["supported"] else "  (below the support floor)"
        print(f"  {_rid:12s} {_cell['n']:5d} decisions  {_cell['rate']:6.1%}{_flag}")
    for _rec in sorted(rep["by_outcome"]):
        _cell = rep["by_outcome"][_rec]
        print(f"  model said {_rec:8s} {_cell['n']:5d}  overridden {_cell['rate']:6.1%}")


_try("exercise 2", _check_override_rates)

## 5. Exercise 3 — time to decision, and the difference between a broken clock and a fast reviewer

Article 26(2) requires the deployer to assign oversight to people with the necessary
competence, training and authority. A log cannot see competence. What it can see is how long
a decision took, and the far tail of that distribution is where a reviewer who is clicking
through a queue shows up. **This is a proxy and the artefact must say so** — a fast decision
on an obvious case is not misconduct.

The trap is the sign. Three decisions here recorded an end before their start. A duration of
minus forty seconds is not a reviewer who was too fast; it is a clock that stepped backwards,
and it belongs in its own list with its own remedy. Include it in the distribution and the
percentiles quietly move; take its absolute value and you have invented a forty-second
review that never happened.

<details><summary>💡 Hint 1 — what to think about</summary>

A duration is missing, negative or real, and only one of those three is evidence about a
reviewer. Where does each kind go before any percentile is taken? Notice too that a negative
number sits below every floor, so the order in which you test things decides whether a
broken clock is reported as a rushed person.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Walk the ids in sorted order and put each record in one of three bins: no duration (count
it), negative (collect its id) or real. Only the real ones feed the list of durations, the
fast list — strictly under the `floor_seconds` argument, not the module constant — and the
list kept for each distinct verifier. Take percentiles and medians with numpy's default
interpolation, and return None for them when there is nothing to take them over.
</details>

In [ ]:
def time_to_decision(view: dict, floor_seconds: float = MIN_PLAUSIBLE_SECONDS) -> dict:
    """The time-to-decision distribution, with the clock anomalies separated out.

    Walk the records in sorted decision_id order. A record with `duration_seconds is None`
    counts in "incomplete" and contributes nothing. A record with a NEGATIVE duration goes in
    "anomalous" and contributes nothing — not to the percentiles, not to "fast", not to any
    reviewer's median. Everything else is a real duration.

    "n"            how many real durations there are
    "median", "p10", "p90"   float(np.percentile(durations, q)) with numpy's default linear
                   interpolation, or None when there are no durations
    "fast"         decision ids whose duration is strictly below floor_seconds, sorted
    "anomalous"    decision ids whose duration is negative, sorted
    "incomplete"   how many records had no duration at all
    "by_reviewer"  {reviewer_id: {"n", "median"}} over each reviewer's own real durations, one
                   entry per decision per DISTINCT verifier

    Returns: dict with keys "n", "median", "p10", "p90", "fast", "anomalous", "incomplete",
    "by_reviewer".

    Example:
        >>> rep = time_to_decision(decision_view(LOG))
        >>> rep["anomalous"]
        ['D-2211', 'D-2733', 'D-3099']
        >>> rep["n"], len(rep["fast"])
        (1351, 14)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_time_to_decision() -> None:
    view = decision_view(LOG)
    rep = time_to_decision(view)
    assert rep["anomalous"] == ["D-2211", "D-2733", "D-3099"], (
        f"anomalous came back {rep['anomalous']} — these are the three decisions whose "
        "duration_seconds is negative, and they must be sorted"
    )
    assert rep["incomplete"] == 7 and rep["n"] == 1351, (
        f"n/incomplete came back {(rep['n'], rep['incomplete'])}, expected (1351, 7): "
        f"{N_DECISIONS} decisions, 7 with no duration, 3 with a negative one"
    )
    assert not set(rep["fast"]) & set(rep["anomalous"]), (
        "a decision appears in BOTH fast and anomalous — a negative duration is not a fast "
        "review, and it has to leave before the floor is applied"
    )
    tighter = time_to_decision(view, floor_seconds=60)
    assert len(tighter["fast"]) > len(rep["fast"]), (
        "raising floor_seconds did not change the fast list — floor_seconds is a parameter, "
        "not the constant MIN_PLAUSIBLE_SECONDS"
    )
    assert rep["by_reviewer"]["u:marlies"]["median"] < \
        rep["by_reviewer"]["u:tomasz"]["median"] / 2, (
        "u:marlies's median time to decision should be well under half u:tomasz's in this log"
    )
    print(f"{rep['n']} timed decisions · p10 {rep['p10']:.0f}s · median {rep['median']:.0f}s "
          f"· p90 {rep['p90']:.0f}s · {len(rep['fast'])} under {MIN_PLAUSIBLE_SECONDS}s · "
          f"{len(rep['anomalous'])} clock anomalies · {rep['incomplete']} never closed")
    for _rid in sorted(rep["by_reviewer"]):
        _cell = rep["by_reviewer"][_rid]
        print(f"  {_rid:12s} {_cell['n']:5d} decisions  median {_cell['median']:7.1f}s")


_try("exercise 3", _check_time_to_decision)

## 6. Exercise 4 — four eyes, counted as people rather than as entries

Article 14(5): no action on the basis of an identification "unless that identification has
been separately verified and confirmed by at least two natural persons". Two *natural
persons*. Not two entries in a list, not two clicks, not one person who submitted the form
twice because the first submit timed out.

The rule reaches only the decisions whose `identification_basis` is `remote_biometric` here.
Everything else is out of scope and is reported as out of scope rather than as compliant,
because "every other decision passed the two-person rule" is a sentence about a rule that was
never applied to them.

Three ways to fail it, and they need different remedies, so they get different reasons.

<details><summary>💡 Hint 1 — what to think about</summary>

Article 14(5) counts natural persons. A list holding one person twice has two entries and one
person: which of your tests can tell those apart? And before you count anyone, ask which
decisions the rule reaches at all — a decision outside its scope is neither compliant nor a
violation, and it is reported as neither.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Walk the ids in sorted order. A record whose identification basis is not the `basis`
argument is only counted as out of scope. For the rest, count distinct verifier ids; when
that is under two, look at the raw list to choose the reason, testing the docstring's three
in order — nobody at all, then one person entered more than once, then one person once. The
rate is compliant over in scope, and the verdict follows from whether any violation exists.
</details>

In [ ]:
def four_eyes_report(view: dict, basis: str = "remote_biometric") -> dict:
    """Article 14(5) compliance over the decisions the rule actually reaches.

    Walk the records in sorted decision_id order. A record whose identification_basis is not
    `basis` counts in "out_of_scope" and is otherwise ignored. For the rest, count DISTINCT
    verifier ids:

      2 or more distinct   -> compliant
      0 verifier ids       -> violation, reason "no_verification"
      1 distinct but 2+ entries -> violation, reason "duplicate_verifier"
      1 distinct, 1 entry  -> violation, reason "single_verifier"

    Those three reasons are in precedence order and they are not interchangeable:
    `no_verification` is a missing control, `duplicate_verifier` is a control that fired twice
    for one person, and `single_verifier` is a control that fired once. An implementation that
    tests `len(verifier_ids) >= 2` passes every duplicate in this log.

    "in_scope"      how many records the rule reaches
    "compliant"     their ids, sorted
    "violations"    list of (decision_id, reason) tuples, sorted by decision_id
    "rate"          len(compliant) / in_scope, or None when in_scope is 0
    "out_of_scope"  how many records the rule does not reach
    "verdict"       "pass" when there are no violations, else "fail"

    Returns: dict with keys "in_scope", "compliant", "violations", "rate", "out_of_scope",
    "verdict".

    Example:
        >>> rep = four_eyes_report(decision_view(LOG))
        >>> rep["in_scope"], len(rep["violations"]), rep["verdict"]
        (273, 39, 'fail')
        >>> sorted({reason for _, reason in rep["violations"]})
        ['duplicate_verifier', 'no_verification', 'single_verifier']
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_four_eyes() -> None:
    rep = four_eyes_report(decision_view(LOG))
    assert rep["in_scope"] == 273 and rep["out_of_scope"] == 1088, (
        f"in_scope/out_of_scope came back {(rep['in_scope'], rep['out_of_scope'])}, expected "
        "(273, 1088) — only the remote_biometric decisions are reached by Article 14(5)"
    )
    reasons = {}
    for _, reason in rep["violations"]:
        reasons[reason] = reasons.get(reason, 0) + 1
    assert reasons.get("duplicate_verifier") == 12, (
        f"duplicate_verifier came back {reasons.get('duplicate_verifier')}, expected 12 — "
        "count distinct persons, not list entries; len(verifier_ids) >= 2 passes all twelve"
    )
    assert reasons.get("single_verifier") == 19 and reasons.get("no_verification") == 8, (
        f"the reason counts came back {reasons}, expected 19 single_verifier and "
        "8 no_verification"
    )
    assert rep["violations"] == sorted(rep["violations"]), "violations must be sorted by id"
    assert rep["verdict"] == "fail"
    print(f"Article 14(5) reaches {rep['in_scope']} decisions · {len(rep['compliant'])} "
          f"compliant ({rep['rate']:.1%}) · {len(rep['violations'])} violations "
          f"· {rep['out_of_scope']} out of scope")
    for _reason in sorted(reasons):
        print(f"  {_reason:22s} {reasons[_reason]:4d}")
    _view = decision_view(LOG)
    _naive = sum(1 for did, _ in rep["violations"]
                 if len(_view[did]["verifier_ids"]) >= 2)
    print(f"\n{_naive} of those violations have two or more ENTRIES in verifier_ids and would "
          "have been reported compliant by a length test")


_try("exercise 4", _check_four_eyes)

## 7. Exercise 5 — automation bias

Here is the measurement the module exists for. Article 14(4)(b) names automation bias as the
thing the overseer must remain aware of. You cannot measure awareness. You can measure the
shape it leaves: **override rate as a function of the model's own confidence.**

A reviewer who overrides less often when the model is more confident is not, on its own,
doing anything wrong — that is what a well-calibrated human working with a decent model looks
like. The pathology is the *level* the curve lands on. A reviewer whose override rate falls
to one per cent on high-confidence scores has stopped being a second opinion.

So the verdict needs two comparisons, and one floor:

* **sensitivity** = override rate below `CONFIDENCE_SPLIT` minus override rate at or above it,
  **signed** — a reviewer who overrides *more* when the model is confident has a different
  problem, not this one;
* that sensitivity compared against **this reviewer's own** minimum detectable difference,
  so a small sample cannot manufacture a finding;
* and the high-confidence rate compared against `HIGH_CONFIDENCE_FLOOR`.

<details><summary>💡 Hint 1 — what to think about</summary>

The status ladder has four rungs and their order is the lesson. What has to be true before
any rate is compared at all? Once a sample can carry a rate, is a decline smaller than what
THIS reviewer's own sample could detect evidence of anything? And why must the sensitivity
keep its sign — what would an absolute value make of a reviewer who overrides MORE when the
model is confident?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Skip any record missing an outcome, a recommendation or a confidence. For each distinct
verifier on the rest, tally decisions and overrides per confidence bin and per half; the
high half starts at `split`, inclusive. Per reviewer, stop at the support rung if the
smaller half is under the floor. Otherwise take low rate minus high rate, signed, work out
the detectable floor from the rate pooled over both halves and the two half sizes, and walk
the remaining rungs in order. Rank findings by sensitivity, largest first, ties by id.
</details>

In [ ]:
def min_detectable_difference(p: float, n_a: int, n_b: int) -> float:
    """P01-L04's two-proportion floor, given back to you here. Not graded.

    The smallest difference in rate that a two-sided 95% test at 80% power could distinguish
    from zero, given a pooled rate `p` and group sizes `n_a`, `n_b`.

    Example:
        >>> round(min_detectable_difference(0.25, 200, 200), 4)
        0.1213
    """
    if n_a <= 0 or n_b <= 0:
        return 1.0
    p = min(max(float(p), 0.0), 1.0)
    return min(1.0, (Z_ALPHA_95 + Z_BETA_80)
               * math.sqrt(p * (1 - p) * (1 / n_a + 1 / n_b)))


# Why the floor cannot be a constant. The same decline in override rate is a finding on one
# sample and noise on another, and the two figures below are computed rather than chosen.
_DEMO_DECLINE = 0.15
for _n in (40, 400):
    _floor = min_detectable_difference(0.30, _n, _n)
    _verdict = "a finding" if _DEMO_DECLINE >= _floor else "not detectable here"
    print(f"{_n:4d} decisions a side, pooled rate 0.30 -> smallest detectable decline "
          f"{_floor:.3f}; a decline of {_DEMO_DECLINE:.2f} is {_verdict}")
print(f"the two floors differ by a factor of "
      f"{min_detectable_difference(0.30, 40, 40) / min_detectable_difference(0.30, 400, 400):.2f}"
      ", which is why one constant cannot serve both")


def automation_bias(view: dict, bins: tuple = CONFIDENCE_BINS,
                    min_support: int = MIN_SUPPORT, split: float = CONFIDENCE_SPLIT,
                    high_confidence_floor: float = HIGH_CONFIDENCE_FLOOR) -> dict:
    """Override rate as a function of the model's confidence, per reviewer.

    A record contributes only when it has an outcome, a recommendation AND a confidence; it
    contributes to each DISTINCT verifier on it. `confidence_bin(confidence, bins)` gives the
    bin; `confidence >= split` puts it in the high half.

    For each reviewer, in sorted id order, build:
      "n", "n_low", "n_high"     decision counts overall and per half
      "low_rate", "high_rate"    override rate in each half, or None when that half is empty
      "sensitivity"              low_rate - high_rate, SIGNED, or None when unsupported
      "min_detectable"           min_detectable_difference(pooled rate over both halves,
                                 n_low, n_high), or None when unsupported
      "per_bin"                  one {"n", "n_overrides", "rate"} per bin, rate None when n is 0
      "status", by this ladder, in this order:
          min(n_low, n_high) < min_support   -> "insufficient_support"
          sensitivity < min_detectable       -> "confidence_independent"
          high_rate <= high_confidence_floor -> "automation_bias"
          otherwise                          -> "confidence_sensitive"

    The order of the second and third rungs matters. A reviewer who overrides 3% of
    low-confidence scores and 2% of high-confidence ones has a high_rate under the floor and a
    sensitivity under their own detectable difference: their rate does not respond to the model
    at all, so it is not automation bias, and this ladder calls them confidence_independent.
    That reviewer IS a finding — but it is a finding about the LEVEL, which override_rates
    reports, not about the slope. Test the sensitivity first or you file one as the other.

    "findings" are the automation_bias reviewers, sorted by sensitivity DESCENDING then by id.
    "unsupported" are the insufficient_support reviewers, sorted. "verdict" is "fail" when
    findings is non-empty, else "pass".

    Returns: dict with keys "by_reviewer", "findings", "unsupported", "verdict".

    Example:
        >>> rep = automation_bias(decision_view(LOG))
        >>> rep["findings"], rep["unsupported"], rep["verdict"]
        (['u:marlies'], ['u:pieter'], 'fail')
        >>> rep["by_reviewer"]["u:tomasz"]["status"]
        'confidence_independent'
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_automation_bias() -> None:
    rep = automation_bias(decision_view(LOG))
    assert set(rep) == {"by_reviewer", "findings", "unsupported", "verdict"}
    assert rep["unsupported"] == ["u:pieter"], (
        f"unsupported came back {rep['unsupported']} — u:pieter has fewer than MIN_SUPPORT "
        "decisions in at least one half, so neither half can carry a rate"
    )
    assert rep["findings"] == ["u:marlies"], (
        f"findings came back {rep['findings']}, expected ['u:marlies'] — she is the only "
        "reviewer whose high-confidence rate is at or under HIGH_CONFIDENCE_FLOOR with a "
        "sensitivity clearing her own detectable floor"
    )
    ines = rep["by_reviewer"]["u:ines"]
    assert ines["status"] == "confidence_sensitive", (
        f"u:ines came back {ines['status']!r} — her rate declines as much as u:marlies's does, "
        "but it lands at a level where she is still disagreeing with the model. Declining is "
        "not the pathology; collapsing is"
    )
    tomasz = rep["by_reviewer"]["u:tomasz"]
    assert tomasz["status"] == "confidence_independent" and tomasz["sensitivity"] < 0.10, (
        f"u:tomasz came back {tomasz['status']!r} with sensitivity {tomasz['sensitivity']} — "
        "his override rate does not respond to the model's confidence, which is what the "
        "status name says and all it says"
    )
    print(f"{'reviewer':12s} {'n':>5s} {'low':>7s} {'high':>7s} {'sens':>7s} {'floor':>7s}  "
          "status")
    for _rid, _cell in rep["by_reviewer"].items():
        _s = "     —" if _cell["sensitivity"] is None else f"{_cell['sensitivity']:7.3f}"
        _f = "     —" if _cell["min_detectable"] is None else f"{_cell['min_detectable']:7.3f}"
        print(f"{_rid:12s} {_cell['n']:5d} {_cell['low_rate']:7.3f} {_cell['high_rate']:7.3f} "
              f"{_s} {_f}  {_cell['status']}")


_try("exercise 5", _check_automation_bias)

Verdicts are easy to disagree with and easy to ignore. The curve is not. The cell below
prints each reviewer's override rate per confidence bin as a bar, and the difference between
a reviewer who is overseeing and one who is confirming is something you can see from across
the room.

In [ ]:
def _plot_bias_curves() -> None:
    rep = automation_bias(decision_view(LOG))
    labels = [f"{CONFIDENCE_BINS[k]:.2f}-{min(CONFIDENCE_BINS[k + 1], 1.0):.2f}"
              for k in range(len(CONFIDENCE_BINS) - 1)]
    print("override rate by model confidence — one bar per bin, 40 columns = 50%\n")
    for rid, cell in rep["by_reviewer"].items():
        print(f"{rid}  ({cell['status']}, n={cell['n']})")
        for label, b in zip(labels, cell["per_bin"]):
            rate = b["rate"]
            if rate is None:
                print(f"   {label}  {'(no decisions)':>8s}")
                continue
            print(f"   {label}  {rate:6.1%}  n={b['n']:4d}  " + "#" * int(round(rate * 80)))
        print()


_try("bias curves", _plot_bias_curves, needs=("exercise 1", "exercise 5"))

## 8. Exercise 6 — the gate

Measuring is half the artefact. Article 14(5) does not ask you to *report* that an action was
taken without two verifications; it says no action shall be taken. So the last thing to build
is the control: a function that looks at one decision and its policy and says whether the
action may proceed.

Four reasons, and the order of two of them matters. A record whose duration is negative gets
`clock_anomaly` and **not** `reviewed_too_fast`: minus forty seconds is less than ten, so a
naive comparison reports a reviewer who rushed, which is a false accusation against a person
and a missed bug in a clock. The fourth reason, `self_verification`, is **this lesson's own
rule and not the Act's** — but "separately verified" is hard to read as satisfied by the
person who submitted the case.

<details><summary>💡 Hint 1 — what to think about</summary>

A negative duration is below every floor, so the naive comparison accuses a named person of
rushing a case whose clock was broken: which test has to come first? And every threshold
here — the verifier count per basis, the seconds floor, both switches — lives in the policy
you are handed. What does your gate do when a caller hands it a different one? For the
report: can one blocked decision carry more than one reason?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

In `oversight_gate`, look the required count up by the record's identification basis,
defaulting to one, and compare it with the number of DISTINCT ids. Treat the duration as
three cases — missing (no reason), negative (the clock reason and nothing else), otherwise
too fast only when strictly under the policy's floor — and read both switches from the
policy. Sort the reasons; `allowed` is just "no reasons". In `gate_report`, walk the ids in
sorted order and add one to every reason each refused decision carries.
</details>

In [ ]:
def oversight_gate(record: dict, policy: dict = POLICY) -> dict:
    """Decide whether one decision's action may proceed under `policy`.

    Collect reasons, each at most once:
      "insufficient_verifiers"  the number of DISTINCT verifier ids is below
                                policy["min_distinct_verifiers"] for this record's
                                identification_basis; a basis the policy does not mention
                                requires 1
      "no_outcome"              policy["require_outcome"] and the record has no outcome
      "clock_anomaly"           duration_seconds is not None and is negative
      "reviewed_too_fast"       duration_seconds is not None, is NOT negative, and is strictly
                                below policy["min_seconds"]
      "self_verification"       policy["forbid_self_verification"] and submitted_by is one of
                                the verifier ids

    Returns: {"decision_id": ..., "allowed": bool, "reasons": sorted list of reason strings}.
    "allowed" is True exactly when reasons is empty.

    Example:
        >>> view = decision_view(LOG)
        >>> oversight_gate(view["D-2211"])["reasons"]
        ['clock_anomaly']
        >>> oversight_gate(view["D-2000"])["allowed"]
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


def gate_report(view: dict, policy: dict = POLICY) -> dict:
    """Run `oversight_gate` over every record and report what it would have blocked.

    Walk the records in sorted decision_id order.

    "n"                how many records there are
    "blocked"          how many the gate refused
    "allowed"          n - blocked
    "blocked_rate"     blocked / n, or None when n is 0
    "blocked_ids"      the refused ids, sorted
    "by_reason"        {reason: count} — a decision blocked for two reasons counts in BOTH, so
                       these need not sum to "blocked". Reasons with no decisions are absent.
    "in_scope_blocked" how many of the refused ones are remote_biometric decisions

    Returns: dict with keys "n", "blocked", "allowed", "blocked_rate", "blocked_ids",
    "by_reason", "in_scope_blocked".

    Example:
        >>> rep = gate_report(decision_view(LOG))
        >>> rep["n"], rep["blocked"], rep["in_scope_blocked"]
        (1361, 83, 44)
        >>> rep["by_reason"]["clock_anomaly"]
        3
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_gate() -> None:
    view = decision_view(LOG)
    anomaly = oversight_gate(view["D-2211"])
    assert "clock_anomaly" in anomaly["reasons"], (
        f"D-2211 came back {anomaly['reasons']} — its duration is negative"
    )
    assert "reviewed_too_fast" not in anomaly["reasons"], (
        "D-2211 was reported as reviewed too fast. Its duration is negative, which is less "
        "than min_seconds on any naive comparison — and it is a broken clock, not a rushed "
        "reviewer. Test the sign before you test the floor"
    )
    rep = gate_report(view)
    assert rep["n"] == N_DECISIONS and rep["allowed"] == rep["n"] - rep["blocked"]
    assert rep["blocked"] == 83, f"blocked came back {rep['blocked']}, expected 83"
    assert rep["by_reason"]["insufficient_verifiers"] == 50, (
        f"insufficient_verifiers came back {rep['by_reason']['insufficient_verifiers']}, "
        "expected 50 — remote_biometric decisions need two DISTINCT persons, credit_file ones "
        "need one"
    )
    assert rep["by_reason"]["self_verification"] == 16, (
        f"self_verification came back {rep['by_reason']['self_verification']}, expected 16 — "
        "the submitter appears among the verifier ids on sixteen decisions here"
    )
    assert sum(rep["by_reason"].values()) > rep["blocked"], (
        "the reason counts sum to no more than the blocked count, so you are counting one "
        "reason per decision; a decision can fail the policy in more than one way"
    )
    loose = gate_report(view, {**POLICY, "forbid_self_verification": False})
    assert loose["blocked"] < rep["blocked"], (
        "turning forbid_self_verification off changed nothing — read the flags out of the "
        "policy argument rather than hard-coding them"
    )
    print(f"the gate blocks {rep['blocked']} of {rep['n']} decisions ({rep['blocked_rate']:.1%}), "
          f"{rep['in_scope_blocked']} of them Annex III point 1(a)")
    for _reason in sorted(rep["by_reason"], key=lambda r: (-rep["by_reason"][r], r)):
        print(f"  {_reason:24s} {rep['by_reason'][_reason]:4d}")


_try("exercise 6", _check_gate)

## 9. The artefact — an oversight plan with measured baselines

An oversight plan that says "all decisions are reviewed by a trained caseworker" cannot be
re-run next quarter and diffed. This one can. It carries the policy it enforces, the
baselines the log actually supports, the reviewers whose behaviour the log cannot evaluate,
and — the part most plans omit — the duties it did **not** measure.

In [ ]:
def oversight_plan(log: list = LOG, policy: dict = POLICY, as_of: date = AS_OF) -> dict:
    """Assemble the oversight plan. Given to you — it is the six measurements in a row."""
    view = decision_view(log)
    overrides = override_rates(view)
    timing = time_to_decision(view)
    four_eyes = four_eyes_report(view)
    bias = automation_bias(view)
    gate = gate_report(view, policy)
    return {"system_id": SYSTEM_ID, "as_of": as_of.isoformat(), "decisions": len(view),
            "policy": policy,
            "baselines": {"override_rate": overrides["overall"]["rate"],
                          "median_seconds": timing["median"],
                          "four_eyes_rate": four_eyes["rate"],
                          "blocked_rate": gate["blocked_rate"]},
            "verdicts": {"four_eyes": four_eyes["verdict"], "automation_bias": bias["verdict"]},
            "findings": {"automation_bias": bias["findings"],
                         "not_evaluable": bias["unsupported"],
                         "four_eyes_violations": len(four_eyes["violations"]),
                         "clock_anomalies": timing["anomalous"],
                         "never_reviewed": overrides["unverified_decided"]},
            "not_measured": [d for d, m in MEASURED_BY.items() if not m]}


def _show_plan() -> None:
    plan = oversight_plan()
    b = plan["baselines"]
    print(f"OVERSIGHT PLAN · {plan['system_id']} · as of {plan['as_of']} · "
          f"{plan['decisions']} decisions\n")
    print(f"  override rate              {b['override_rate']:.1%}")
    print(f"  median time to decision    {b['median_seconds']:.0f}s")
    print(f"  Article 14(5) compliance   {b['four_eyes_rate']:.1%}")
    print(f"  blocked by the gate        {b['blocked_rate']:.1%}")
    print(f"\n  verdicts   {plan['verdicts']}")
    for name, value in plan["findings"].items():
        print(f"  {name:24s} {value}")
    print("\n  duties this plan does NOT measure, because a log cannot see them:")
    for duty in plan["not_measured"]:
        print(f"    {duty:30s} {DUTIES[duty]}")


_try("oversight plan", _show_plan, needs=tuple(_EXERCISES))   # it is all six in a row

## 10. Common mistakes

- **Deduplicating verifiers in the fold.** The single most effective way to defeat a
  four-eyes rule is one person clicking twice, and a view that collapses the list at the
  source makes that defect unobservable everywhere downstream.
- **Reading `len(verifier_ids) >= 2` as "two natural persons".** In this log that passes
  every duplicate. The article says persons; count persons.
- **Applying 14(5) to the whole log.** It reaches the Annex III point 1(a) systems, and it
  has a law-enforcement carve-out. A plan that demands two signatures everywhere buys nothing
  and is abandoned in a quarter.
- **abs() on a negative duration.** It converts a broken clock into a plausible review time,
  and the bug never surfaces again.
- **Calling every decline in override rate automation bias.** A reviewer who disagrees less
  with a confident model may simply be right. What is not defensible is a rate that lands at
  one per cent: at that level the reviewer adds no information to the system's output.
- **A fixed sensitivity threshold.** The cell above `automation_bias` prints the detectable
  floor at two sample sizes and the factor between them. A constant chosen for one of those
  samples publishes noise on the other, or discards a real decline. Compare each reviewer
  against what their own sample could have detected.
- **Reading `confidence_independent` as a clean bill of health.** It means one thing: this
  reviewer's override rate does not move with the model's confidence. A reviewer who
  overrides two per cent of everything gets that label too, and they are the worst reviewer
  on the page. The finding against them is the LEVEL in `override_rates`, not the slope
  here, which is why the two reports are read together and why the status is not called
  "independent".
- **Dropping the reviewer below the support floor.** `u:pieter` is not evaluable, which is
  itself a finding: this deployer has a reviewer nobody can tell anything about.
- **Counting unreviewed decisions nowhere.** They belong to no reviewer, so a per-reviewer
  report loses them silently. They are the worst number on the page.

The cell below strips the automation-bias curve out of the report and prints what is left,
which is what most human-oversight plans actually contain.

In [ ]:
def _show_without_the_curve() -> None:
    rep = override_rates(decision_view(LOG))
    print("the same reviewers with only their headline override rate:\n")
    for rid in sorted(rep["by_reviewer"]):
        cell = rep["by_reviewer"][rid]
        print(f"  {rid:12s} {cell['n']:5d} decisions   {cell['rate']:6.1%}")
    bias = automation_bias(decision_view(LOG))
    worst = bias["findings"][0] if bias["findings"] else None
    print(f"\nNothing here is alarming, and {worst} is not even the highest number on the "
          "page.")
    print(f"Split her by confidence and her high-confidence override rate is "
          f"{bias['by_reviewer'][worst]['high_rate']:.1%}. That is the module.")


_try("without the curve", _show_without_the_curve,
     needs=("exercise 1", "exercise 2", "exercise 5"))

## 11. Self-check

1. The curve above shows `u:marlies` overriding most low-confidence scores and almost no
   high-confidence ones, and `u:tomasz` at much the same rate throughout. What does the log
   support?
   - (a) marlies is biased and tomasz is not
   - (b) her rate collapses as confidence rises and his does not — whether she is deferring
         to a model that is right is a question the log cannot answer
   - (c) tomasz is worse, because his rate ignores the model's confidence
   - (d) neither is overseeing, because both override less than half the time

2. An Annex III point 1(a) decision records `verifier_ids: ["u:marlies", "u:marlies"]`. Under
   Article 14(5) it is:
   - (a) compliant — two identifiers were recorded
   - (b) compliant if the two entries carry different timestamps
   - (c) not compliant — the rule counts natural persons, and that is one person
   - (d) out of scope, because the identification was not contested

3. Which systems does the two-person rule in Article 14(5) reach?
   - (a) every high-risk system in Annex III
   - (b) the Annex III point 1(a) remote biometric identification systems, minus the law
         enforcement, migration, border control and asylum carve-out
   - (c) every system that processes personal data
   - (d) only systems whose provider chose a notified body

4. A decision's `use_end` is forty seconds before its `use_start`. Your report should:
   - (a) call it a clock anomaly, drop it from the distribution, and never call it "reviewed
         too fast" — a negative duration is not evidence about the reviewer
   - (b) include -40 in the distribution; the data is the data
   - (c) take the absolute value, so it contributes forty seconds
   - (d) drop it silently

5. The report labels `u:tomasz` `confidence_independent` and prints his detectable floor
   beside it. That label means:
   - (a) his override rate is provably unrelated to the model's confidence
   - (b) he overrides more often than every other reviewer
   - (c) his curve shows no decline larger than the smallest one this many of his decisions
         could have found — about his log and about the test's sensitivity at once
   - (d) the test failed and returned a default

In [ ]:
# Salted hashes of the answers, not the answers. Nothing here tells you which letter is right.
_SELF_CHECK_KEY = {
    1: "7dc18b059e48e0ce",
    2: "2c6b66ba526de0f5",
    3: "71c046491e87b753",
    4: "f0fbe0c8d890a3b2",
    5: "5a76561d92714f73",
}

_SELF_CHECK_HINT = {
    1: "re-read the first two paragraphs of section 7, and ask what the log cannot see.",
    2: "read the reason ladder in the four_eyes_report docstring, and count persons.",
    3: "look at the note directly under the duties table in section 1.",
    4: "run the gate on D-2211 and read which reason it gives, and which it does not.",
    5: "look at what min_detectable is computed from, and what happens to it as n falls.",
}


def check_self_check(answers: dict) -> None:
    """Mark your self-check answers. Pass a dict of question number -> letter.

    Example:
        >>> check_self_check({1: "a"})          # doctest: +SKIP
          q1  not 'a' — re-read the first two paragraphs of section 7 ...
          q2  no answer given
        ...
    """
    right = 0
    for question in sorted(_SELF_CHECK_KEY):
        given = str(answers.get(question, "")).strip().lower()
        digest = hashlib.sha256(f"P01-L05:q{question}:{given}".encode()).hexdigest()[:16]
        if digest == _SELF_CHECK_KEY[question]:
            right += 1
            print(f"  q{question}  correct")
        elif not given:
            print(f"  q{question}  no answer given")
        else:
            print(f"  q{question}  not {given!r} — {_SELF_CHECK_HINT[question]}")
    print(f"\n{len(_SELF_CHECK_KEY)} questions, {right} right")


# Put your own letters in, then run this cell:
# check_self_check({1: "a", 2: "a", 3: "a", 4: "a", 5: "a"})

## What you built, and where it goes next

A decision view that preserves the evidence a naive fold destroys, four measurements that
turn "we have human oversight" into numbers with a denominator you can argue with, and a gate
that refuses an action the policy does not support. That is the object behind the conformity
pack's `human_oversight_plan` — a row that used to be a status word and is now a thing you
re-run against next quarter's log and diff.

The pieces travel. The support floor is P01-L04's, unchanged, and the detectable-difference
formula with it. The sequence-order discipline and the log format are P01-L01's. The gate is
the shape module 7's post-market monitor needs when it decides whether a wobble is an
incident, and the capstone hands you an oversight record where one reviewer approved four
hundred cases in an hour — which is `time_to_decision` and nothing else.

**Again, and finally: this is engineering, not legal advice.**

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_decision_view),
                              ("exercise 2", _check_override_rates),
                              ("exercise 3", _check_time_to_decision),
                              ("exercise 4", _check_four_eyes),
                              ("exercise 5", _check_automation_bias),
                              ("exercise 6", _check_gate)):
            _try(_name, _check)
    _progress_board()
    # A stub nobody has reached yet is not a failure. A check that ran and came back wrong is:
    # in a script or under CI it ends this run non-zero, rather than letting a green exit code
    # paper over it. Inside a notebook kernel the board above has already said so, in a line
    # rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))